### Imports

In [40]:
import os
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

### Read files

In [41]:
X_train = pd.read_csv('X_train.csv',index_col='ROW_ID')
y_train = pd.read_csv('y_train.csv',index_col='ROW_ID')
X_test = pd.read_csv('X_test.csv',index_col='ROW_ID')

### Initial exploration

In [42]:
print(X_train.head(3)) # print the data structure to double check, some NaN are present

               TS     ALLOCATION    RET_20    RET_19    RET_18    RET_17  \
ROW_ID                                                                     
0       DATE_0001  ALLOCATION_01 -0.018192 -0.000306 -0.006881 -0.002393   
1       DATE_0001  ALLOCATION_02 -0.006394 -0.001059  0.001565  0.000033   
2       DATE_0001  ALLOCATION_03 -0.016587 -0.004517 -0.005306  0.004314   

          RET_16    RET_15    RET_14    RET_13  ...  SIGNED_VOLUME_8  \
ROW_ID                                          ...                    
0       0.000507 -0.001270 -0.002539  0.002830  ...         0.818730   
1       0.002829  0.001725  0.000875 -0.002160  ...        -1.390336   
2       0.006471 -0.005868 -0.005030 -0.001488  ...         0.961318   

        SIGNED_VOLUME_7  SIGNED_VOLUME_6  SIGNED_VOLUME_5  SIGNED_VOLUME_4  \
ROW_ID                                                                       
0              0.941014         0.714129        -0.323847         0.525097   
1             -0.651784 

In [43]:
print(X_test.head(3)) # same situations

               TS     ALLOCATION    RET_20    RET_19    RET_18    RET_17  \
ROW_ID                                                                     
527073  DATE_2523  ALLOCATION_01 -0.000523  0.000066 -0.001601  0.002030   
527074  DATE_2523  ALLOCATION_02  0.002868 -0.000870  0.000609  0.003156   
527075  DATE_2523  ALLOCATION_03 -0.000435  0.003141  0.001410 -0.011531   

          RET_16    RET_15    RET_14    RET_13  ...  SIGNED_VOLUME_8  \
ROW_ID                                          ...                    
527073 -0.000137 -0.001506  0.000479 -0.001006  ...        -1.124653   
527074  0.001140  0.000728  0.002079  0.000126  ...        -1.310546   
527075 -0.000685  0.002088 -0.003898  0.008467  ...         1.402281   

        SIGNED_VOLUME_7  SIGNED_VOLUME_6  SIGNED_VOLUME_5  SIGNED_VOLUME_4  \
ROW_ID                                                                       
527073        -1.263611        -1.177919        -1.251840        -1.242161   
527074        -0.925154 

In [44]:
print(y_train.head(3)) # fast print check on target data structure

          target
ROW_ID          
0       0.009210
1       0.000405
2       0.005967


In [45]:
print(X_train.shape, X_test.shape) # look at the shape of the data
print("portfolios:", X_train["ALLOCATION"].nunique(), X_test["ALLOCATION"].nunique()) # 
# we have the same number of portfolio 278 and same number of columns in each database, good.

(527073, 44) (31870, 44)
portfolios: 278 278


In [46]:
X_train["target"] = y_train["target"]      # we attach the next-day return to each row
print("share of up days:", (X_train["target"] > 0).mean().round(4)) # we see how often we have up and down days

share of up days: 0.5072


In [47]:
RET = [f"RET_{i}" for i in range(1, 21)]   # RET_1 = most recent day
# look at the number of Nan, i.e., missing days. We will have to deal with them
print("missing RET values: train", X_train[RET].isna().sum().sum(), " test", X_test[RET].isna().sum().sum()) 

missing RET values: train 262  test 294


### Data cleaning & sorting

Here, the idea is relatively simple. We are looking at returns in 20 days window and trying to predict the sign (i.e., up or down) of the next day. For each portfolio, of course.

Now, we note that for each portfolio tomorrow's row in our database (if present) should hold today's return shifted by one slot. To wit, today's RET_1 $\sim$ RET_2 of tomorrow, today's RET_2 $\sim$ RET_3 of tomorrow, and so on. The match is assumed to be approximate, not exact, due to the drift in the portfolio allocation through each day. 

Therefore, we can try to build longer windows of data (to help us test the model) by looking at rows whose shifted windows matches another one. We expect to find some, but not all.

In [48]:
# We begin by looking at a single portfolio
overlap = 15 # number of days we compare (to be checked/tuned later)
g = X_train[X_train["ALLOCATION"] == "ALLOCATION_01"]
R = np.nan_to_num(g[RET].to_numpy()) # the rows, i.e., the dates of the portfolio

today = R[:,0:overlap]        # RET 1 - 15
tomorrow = R[:,1:overlap+1]   # RET 2 - 16

# Since we expect drifts, a good practice is to normalise our windows
epsilon = 1e-12  # to avoid divergences
today = today/(np.linalg.norm(today, axis =1, keepdims = True) + epsilon)
tomorrow = tomorrow/(np.linalg.norm(tomorrow, axis =1, keepdims = True) + epsilon)

# we then extract the 2 rows whose shifted window is closest

dist, idx = cKDTree(tomorrow).query(today, k =2)

# if we have a clear winner
ratio = 0.3     # to be checked/tuned later
clear = dist[:,0] < ratio*dist[:,1]
print("rows:", len(g), "| rows with a clear next-day match:", clear.sum())


# let us look at a few matches to make sure 
m = pd.DataFrame({
    "date": g["TS"].values,
    "next_date": g["TS"].values[idx[:, 0]],
    "RET_1 today": g["RET_1"].values,
    "RET_2 of match": g["RET_2"].values[idx[:, 0]],
    "clear": clear,
})
print(m[m["clear"]].head(10))

rows: 1431 | rows with a clear next-day match: 1131
         date  next_date  RET_1 today  RET_2 of match  clear
0   DATE_0001  DATE_0404     0.003944        0.003131   True
2   DATE_0005  DATE_2141     0.005628        0.006723   True
3   DATE_0007  DATE_1332    -0.003151       -0.005836   True
4   DATE_0010  DATE_0343     0.005724        0.005943   True
5   DATE_0014  DATE_2494    -0.002946       -0.003865   True
6   DATE_0015  DATE_2501    -0.001909       -0.002236   True
8   DATE_0017  DATE_1488    -0.000375        0.001183   True
9   DATE_0019  DATE_2048    -0.001959       -0.003233   True
10  DATE_0023  DATE_1617     0.001064        0.000818   True
11  DATE_0026  DATE_1312     0.000855        0.001245   True


In [ ]:
# We run for one portfolio to check it made sense, now 
# we build a function.

def next_day_votes(X, k=1, overlap = 15, ratio = 0.3):
    votes = []
    for a, g in X.groupby("ALLOCATION"):
        R = np.nan_to_num(g[RET].to_numpy()) 
        original = R[:,0:overlap]/(np.linalg.norm(R[:,0:overlap])+epsilon)      
        shifted  = R[:,k:overlap+k]/(np.linalg.norm(R[:,k:overlap+k])+epsilon)      
        d, j = cKDTree(shifted).query(original, k =2)
        clear = d[:,0] < ratio*d[:,1]
        votes.append(pd.DataFrame({"ALLOCATION": a, "TS":g["TS"].values[clear],"NEXT_TS": g["TS"].values[j[clear,0]]}))
    return pd.concat(votes, ignore_index = True)

votes = next_day_votes(X_train, k=1, overlap=overlap, ratio= ratio)

print("rows:", len(X_train), "| clear votes:", len(votes), f"({len(votes) / len(X_train):.1%})")
print("dates that received at least one vote:", votes["TS"].nunique(), "of", X_train["TS"].nunique())
print(votes.head())

TypeError: query() takes at least 1 positional argument (0 given)